In [1]:
import pandas as pd

In [2]:
#On va explorer notre dataset en chargeant les données en visualisant( les colonnes et les premières lignes), en les comprenant puis les uniformiser et les nettoyer pour avoir un dataset conforme

In [3]:
df = pd.read_csv("../data/raw/dataset.csv", sep=';')
df.head()

,sentence_1,sentence_2,similarity_score
0,Chemicals for use in agriculture,Chemicals for the protection of plants [other ...,1.0
1,Telecommunication services,Provision of wireless application protocol ser...,1.0
2,Clothing,Gloves with conductive fingertips that may be ...,1.0
3,Telecommunication services,"Communication services, namely, electronic tra...",1.0
4,Hand tools and implements [hand-operated],Parts and fittings for arbors bits chucks dril...,1.0


D'après le sujet, il s'agit de construire un systeme de scoring d'une paire de texte. Dans notre dataset, nous avons une collection de paires de texte: sentence1, sentence2 avec un score qui leur est associé: similarityscore(qui varie de 0 à 1). Plus le similarityscore est élevé plus il y a une correspondance entre les paires de textes. Et donc les paires de modèles seront utilisés pour améliorer des modèles déja existants.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34560 entries, 0 to 34559
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sentence_1        34560 non-null  object 
 1   sentence_2        34560 non-null  object 
 2   similarity_score  34560 non-null  float64
dtypes: float64(1), object(2)
memory usage: 810.1+ KB


On a un dataset avec 34560 lignes ce qui est pas mal conséquent avec les types de nos colonnes

In [5]:
#Je vais vérifier si y a des valeures nulles
df.isnull().sum()

sentence_1          0
sentence_2          0
similarity_score    0
dtype: int64

On a aucune valeures manquantes au niveau de nos colonnes ce qui est bien :)


Apres exploration visuel du dataset je vois qu'il ya des crochets et des parentheses qui faut supprimer pour l approche tf-idf
On va maintenant construire notre script de nettoyage de notre dataset et l'enregistrer en sortie dans notre dossier /processed

Notre script de nettoyage fonctionne et le fichier de sortie est enregistré dans le dossier /processed sous le nom "dataset_nettoye"

A savoir:
On a versionné notre dataset et notre model avec DVC.
Maintenant on va automatiser la pipeline afin que cela se fasse automatiquement.

Si y a une modification sur notre dataset de base cela le detecte et reprend le processus de nettoyage

Cela a fonctionné et notre fichier dvc.yaml a été crée avec succès :)

Maintenant on va passer à la partie entrainement de notre model

Pour créer notre model on va se baser sur l'approche Tf-idf qui est de la régression qui est un peu plus complexes que mes projets de classification.

De mon interpretation de l'approche tf-idf, il s'agit de savoir (le nombre de fois qu'un mot apparait dans un document/nombre total de mots dans le document:tf) et (le nombre total de documents/le nombre total de documents qui contient ce mot)

Lorsque quon obtient ces deux rapports il s'agirait de mesurer la valeur tf-idf qui est le produit des deux avec un log népérien sur l'idf. 

La valeur tf-idf nous permet de mesurer l importance du mot dans un doc

Pour faire cette étape, il faut du'on vectorise chaque document et qu'on suive des processus

Prenons l'exemple de notre dataset

In [6]:
df = pd.read_csv("../data/processed/dataset_nettoye.csv", sep=';')
df.head()

,sentence_1,sentence_2,similarity_score
0,chemicals for use in agriculture,chemicals for the protection of plants other t...,1.0
1,telecommunication services,provision of wireless application protocol ser...,1.0
2,clothing,gloves with conductive fingertips that may be ...,1.0
3,telecommunication services,"communication services, namely, electronic tra...",1.0
4,hand tools and implements hand-operated,parts and fittings for arbors bits chucks dril...,1.0


Prenonsl'exemple de la premiere ligne
Nous avons:
1-Chemicals for use in agriculture
2-Chemicals for the protection of plants
Et on on un similarityscore=1 ce qui veut dire que nous deux textes sont trés similaires. Notre model devra donc le deviner.

Je comprends que la relation qui existe entre les texte peut etre une catégorie (par exemple téléphones-> outils technologique) etc..

Voici globalement comment le tf-idf fonctionne ici en prenant notre premiere ligne:
On prend chaque phrase et on enleve les mots inutiles de la phrase

-Chemicals for use in agriculture -> [chemicals, use, agriculture]
-Chemicals for protection of plants -> [chemicals, protection, plants]

On obtient ensuite des vecteurs sous forme de nombre et on calcul la similarité

On va ensuite crée nos deux scripts vectorisation.py pour vectoriser chaque texte de notre dataset et le tf-idf en chargeant les bibliothèques de vectorisation ... etc et modelisation.py pour l'apprentissage et la sauvegarde de notre modele

In [7]:
#Démarrons la vectorisation et l'approche tf-idf

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
df = pd.read_csv("../data/processed/dataset_nettoye.csv", sep=';')

In [10]:
vectorisation_des_phrases = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words="english")
phrases_du_dataset = pd.concat([df["sentence_1"],df["sentence_2"]])
print(phrases_du_dataset[:3])

0    chemicals for use in agriculture
1          telecommunication services
2                            clothing
dtype: object


In [11]:
vectorisation_des_phrases.fit(phrases_du_dataset)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.

In [12]:
noms_de_vectorisation = vectorisation_des_phrases.get_feature_names_out()
print(noms_de_vectorisation[:10])

['abraders' 'abrasive' 'abrasive cloth' 'abrasive paper' 'abrasives'
 'absorbent' 'absorbent articles' 'absorber' 'absorber plungers'
 'absorbers']


In [13]:
#On observe les termes les plus important abrasive cloth , ....
#et les moins importants sont:
print(noms_de_vectorisation[-10:])

['xylol' 'yarns' 'yarns threads' 'yeast' 'yeast leavening' 'yeasts'
 'yoghurt' 'yogurt' 'yogurts' 'yogurts sorbets']


On va maintenant créer notre scripts vectorisation.py et sauveagarder notre model a la fin

In [14]:
import joblib

In [15]:
#on va sauvegarder notre model dans /models
joblib.dump(vectorisation_des_phrases, "../models/vectorisation.pkl")
print("La vectorisation a été éffectué avec succès")

La vectorisation a été éffectué avec succès


Notre model a bien été enregistré dans le dossier /models
Passons maintenant a la modelisation

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
df = pd.read_csv("../data/processed/dataset_nettoye.csv", sep=';')

vectorisation_des_phrases = joblib.load("../models/vectorisation.pkl")

Il s'agirait ici pour nous de prendre nos colonnes séparemment c'est a dire "sentence_1" et "sentence_2" et de calculer la similarité pour chaque ligne c'est a dire pour chaque jeu de phrases

In [18]:
vecteurs_1 = vectorisation_des_phrases.transform(df["sentence_1"])
vecteurs_2 = vectorisation_des_phrases.transform(df["sentence_2"])
#affichons chaque premier vecteur voir
print(vecteurs_1[:1])
print(vecteurs_2[:1])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 88)	0.4267997993384776
  (0, 755)	0.4009238075707081
  (0, 757)	0.527007069387993
  (0, 4724)	0.3146204614660667
  (0, 4725)	0.5295087778777603
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 755)	0.36216937839274893
  (0, 1905)	0.3957102015544637
  (0, 2143)	0.39240517773644035
  (0, 2313)	0.5128707143604981
  (0, 3266)	0.407074042147728
  (0, 3484)	0.3598879114208064


In [19]:
#On va essayer de le visualiser avec la premiere ligne juste de la colonne "sentence1" pour voir le mécanisme avec 0 l'index de notre premiere phrase
phrasenumero1 = df["sentence_1"].iloc[0]
transformationchiffrephrase1 = vecteurs_1[0]

#Voici la visualisation de la transformation
print("Voici la visualisation de la transformation pour la premiere phrase \n")
print(phrasenumero1)
print(transformationchiffrephrase1)




Voici la visualisation de la transformation pour la premiere phrase 

chemicals for use in agriculture
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 88)	0.4267997993384776
  (0, 755)	0.4009238075707081
  (0, 757)	0.527007069387993
  (0, 4724)	0.3146204614660667
  (0, 4725)	0.5295087778777603


on calcul maintenant le cosine_similare entre deux vecteurs v1 et V2 

In [20]:
similarites = []
for i in range(len(df)):
    score_de_comparaison = cosine_similarity(vecteurs_1[i],vecteurs_2[i])[0][0]
    similarites.append(score_de_comparaison)

#on va essayer d'afficher la similarité de notre premiere lige voir ce que ca donne
print(f"La similarité des deux phrases de la premiere ligne est: {similarites[0]}")
    

La similarité des deux phrases de la premiere ligne est: 0.1452023261707375


Lorsque je le compare a la similarite du dataset cela a l'air insuffisant. Je vais afficher la similarite de la derniere ligne pour voir si tout va bien.

In [21]:
print(f"La similarité des deux phrases de la premiere ligne est: {similarites[-1]}")

La similarité des deux phrases de la premiere ligne est: 0.0


On trouve O cela a l'air logique

On va maintenant passer à l'iniatiatisation de WB pour visualiser les performances de la variation des parametres de mon model

In [22]:
import wandb
import numpy as np

In [23]:
#initialisation de notre wb
wandb.init(
    project="projet_test_technique_mlops_features5000",
    config={
        "model_type": "TF-IDF + Cosine",
        "max_features": 5000,
        "dataset_size": "all"
    }
)
df = pd.read_csv("../data/processed/dataset_nettoye.csv", sep=';')
wandb.config.update({
    "nombredelignes_dataset": len(df),
    "colonnes_dataset": list(df.columns)
})
#on affiche ensuite les logging de notre model
wandb.log({
    "moyenne_similarite": np.mean(similarites),
    "similarite_maximale": np.max(similarites)
})
wandb.finish()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/dide/.netrc.
wandb: Currently logged in as: didymedide0808 (didymedide0808-s) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


moyenne_similarite,▁
similarite_maximale,▁
moyenne_similarite,0.03496
similarite_maximale,1.0


Tout fonctionne s'est parfait je visualise la performance des mes differents models. Maintenant je vais créer mon script modelisation et ensuite l'automatiser avec dvc et aussi enregistrer mon fichier de sortie dans artefact de wandb.
#Très important: apres je pourrer creer un autre model en faisant dabord une vectorisation cette fois ci avec un max_features à 10000 si j'ai le temps pour voir le résultat